# Notebook 06: Normalization and Training Stability

## Why This Matters

Normalization is not a detail — it's a foundational design decision that determines whether your model trains at all. The shift from BatchNorm to LayerNorm enabled transformer training on sequences. The shift from Post-LN to Pre-LN removed the need for learning rate warmup and enabled training of 100+ layer models. The shift from LayerNorm to RMSNorm (LLaMA, Gemma, Mistral) reduced compute and memory for normalization by 10-20%. If you're debugging a transformer that's not converging, normalization placement is one of the first things to check. This notebook builds LayerNorm and RMSNorm from scratch, visualizes what happens to activations across depth with and without normalization, and explains the QK-norm trick used in some frontier models.

## Background

The core challenge in training deep neural networks is keeping activations and gradients in a well-behaved range across all layers. Without normalization:
- **Forward pass:** activations grow (or shrink) exponentially with depth
- **Backward pass:** gradients vanish or explode, making early layers un-trainable

Normalization layers solve this by explicitly rescaling activations at each layer, keeping them in a standard range. The question is: what statistics to normalize over (batch, layer, instance, group), and where to apply normalization (before or after the sublayer)?

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
print("Ready.")

## 1. BatchNorm vs LayerNorm: The Core Distinction

**BatchNorm** (Ioffe & Szegedy, 2015) normalizes over the **batch and spatial dimensions** for each feature:
$$\hat{x}_{b,t,d} = \frac{x_{b,t,d} - \mu_d}{\sigma_d + \epsilon}$$
where $\mu_d$ and $\sigma_d$ are computed over the entire batch and all positions for feature $d$.

**LayerNorm** (Ba et al., 2016) normalizes over the **feature dimension** for each (batch, position):
$$\hat{x}_{b,t} = \frac{x_{b,t} - \mu_{b,t}}{\sigma_{b,t} + \epsilon}$$
where $\mu_{b,t}$ and $\sigma_{b,t}$ are computed over the $d_{\text{model}}$ features for token $(b, t)$.

**Why LayerNorm for sequences:**
1. **Variable sequence lengths:** BatchNorm requires aggregating over all positions; variable-length sequences complicate this
2. **Small batch sizes:** At inference with batch=1, BatchNorm statistics are undefined (must use running statistics, which are often stale)
3. **Autoregressive generation:** Generates one token at a time — batch size is 1, making BatchNorm unusable

**Interview question:** "Can you use BatchNorm in a transformer?"  
**Answer:** Technically yes for fixed-length inputs with large batches, but LayerNorm is universally preferred because it works regardless of batch size and sequence length. The few attempts to use BatchNorm in transformers (e.g., PowerNorm) haven't displaced LayerNorm.

In [ ]:
class LayerNorm(nn.Module):
    """LayerNorm from scratch — normalize over feature dimension."""
    def __init__(self, normalized_shape, eps=1e-6):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = normalized_shape
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps

    def forward(self, x):
        dims = tuple(range(-len(self.normalized_shape), 0))
        mean = x.mean(dim=dims, keepdim=True)
        var = x.var(dim=dims, keepdim=True, unbiased=False)
        x_norm = (x - mean) / (var + self.eps).sqrt()
        return self.gamma * x_norm + self.beta

# Verify
d = 256
ln = LayerNorm(d)
x = torch.randn(8, 20, d) * 10 + 5  # Offset deliberately
out = ln(x)
print(f"Input  — mean: {x.mean().item():.3f}, std: {x.std().item():.3f}")
print(f"Output — mean: {out.mean().item():.6f} (should be ~0)")
print(f"Output — std:  {out.std().item():.6f}  (should be ~1)")

# Show that it normalizes per (batch, position) NOT across batch
means_per_token = out.mean(dim=-1)  # (B, T)
print(f"Per-token mean range: [{means_per_token.min().item():.8f}, {means_per_token.max().item():.8f}]")

## 2. RMSNorm: Remove the Mean, Keep the Scale

RMSNorm (Zhang & Sennrich, 2019) simplifies LayerNorm by removing mean subtraction:

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x) + \epsilon} \cdot \gamma$$
$$\text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^d x_i^2}$$

**Why remove the mean centering?**
1. **Cheaper:** No mean computation needed — ~20-40% faster than LayerNorm on some hardware
2. **Empirically equivalent:** The mean centering in LayerNorm is somewhat redundant when combined with learned $\gamma, \beta$ — the scale parameter can compensate
3. **Cleaner theory:** "Root mean square scaling" is more principled for controlling the L2 norm of representations

**Used in:** LLaMA (all versions), Mistral, Gemma, Falcon, Qwen, and most post-2022 LLMs. If you see a model using `rms_norm_eps` in its config, it's using RMSNorm.

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization — used in LLaMA, Mistral, Gemma."""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        # x: (..., d_model)
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return (x / rms) * self.gamma

# Compare LayerNorm vs RMSNorm behavior
d = 256
ln_norm = LayerNorm(d)
rms_norm = RMSNorm(d)
x = torch.randn(4, 16, d) * 5 + 2

out_ln = ln_norm(x)
out_rms = rms_norm(x)

print("LayerNorm output stats:")
print(f"  mean={out_ln.mean().item():.6f}, std={out_ln.std().item():.6f}")
print("RMSNorm output stats (no mean centering):")
print(f"  mean={out_rms.mean().item():.6f}, std={out_rms.std().item():.6f}")

# Speed comparison
import time
x_large = torch.randn(32, 512, 4096)
n_iter = 100

start = time.perf_counter()
for _ in range(n_iter):
    _ = nn.LayerNorm(4096)(x_large)
ln_time = time.perf_counter() - start

start = time.perf_counter()
for _ in range(n_iter):
    _ = RMSNorm(4096)(x_large)
rms_time = time.perf_counter() - start

print(f"\nLayerNorm: {ln_time*1000:.1f}ms for {n_iter} iters")
print(f"RMSNorm:   {rms_time*1000:.1f}ms for {n_iter} iters")
print(f"RMSNorm speedup: {ln_time/rms_time:.2f}x")

## 3. Pre-LN vs Post-LN: Activation Scale Across Depth

The placement of normalization relative to the residual connection fundamentally changes how activations scale with depth. Let's measure this empirically.

In [ ]:
class PostLNLayer(nn.Module):
    """Post-LN: LN after residual — original Vaswani 2017."""
    def __init__(self, d_model):
        super().__init__()
        self.linear = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.norm(x + self.linear(x))

class PreLNLayer(nn.Module):
    """Pre-LN: LN before sublayer — modern standard."""
    def __init__(self, d_model):
        super().__init__()
        self.linear = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return x + self.linear(self.norm(x))

def measure_activation_stats(LayerClass, d_model=128, n_layers=24, B=4, T=16):
    """Track mean absolute activation and gradient norms across layers."""
    layers = nn.ModuleList([LayerClass(d_model) for _ in range(n_layers)])
    x = torch.randn(B, T, d_model, requires_grad=True)
    act_norms = []
    activations = [x]
    h = x
    for layer in layers:
        h = layer(h)
        act_norms.append(h.detach().abs().mean().item())
        activations.append(h)
    # Compute gradients
    loss = activations[-1].sum()
    loss.backward()
    grad_norm = x.grad.abs().mean().item() if x.grad is not None else 0
    return act_norms, grad_norm

post_acts, post_grad = measure_activation_stats(PostLNLayer)
pre_acts, pre_grad = measure_activation_stats(PreLNLayer)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
layers_x = range(1, len(post_acts)+1)
axes[0].plot(layers_x, post_acts, 'r-o', markersize=5, label='Post-LN')
axes[0].plot(layers_x, pre_acts,  'b-o', markersize=5, label='Pre-LN')
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Mean |activation|")
axes[0].set_title("Activation Scale Across Depth")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(['Post-LN', 'Pre-LN'], [post_grad, pre_grad], color=['red', 'blue'])
axes[1].set_ylabel("Mean |input gradient|")
axes[1].set_title("Input Gradient Magnitude")
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle("Post-LN vs Pre-LN: Activation and Gradient Behavior (24 layers)", fontsize=12)
plt.tight_layout()
plt.savefig("preln_vs_postln.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"Post-LN activation range: {min(post_acts):.3f} – {max(post_acts):.3f}")
print(f"Pre-LN  activation range: {min(pre_acts):.3f} – {max(pre_acts):.3f}")

## 4. QK-Norm: Stabilizing Attention at Scale

Some modern models (Gemma 2, SD3, some ViT variants) apply normalization to query and key vectors before computing attention scores:

$$\text{score}_{ij} = \frac{\text{RMSNorm}(q_i) \cdot \text{RMSNorm}(k_j)}{\sqrt{d_k}}$$

**Why QK-Norm?** At large scale, attention logit growth is a training instability. If $\|q\|$ or $\|k\|$ grows large during training, the attention scores spike, pushing softmax toward one-hot distributions. This causes sharp "attention spikes" that destabilize training. QK-Norm prevents this by bounding the L2 norm of Q and K to 1 (times the learned scale), keeping attention logits bounded regardless of model size or training duration.

**Impact:** Enables more aggressive learning rates, reduces the need for attention clipping, and improves training stability for very large models and long training runs.

In [ ]:
class QKNormAttention(nn.Module):
    """Multi-head attention with QK normalization (Gemma 2 style)."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        # Per-head QK normalization
        self.q_norm = RMSNorm(self.d_k)
        self.k_norm = RMSNorm(self.d_k)

    def split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        Q = self.split_heads(self.W_q(x))  # (B, h, T, d_k)
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        # QK normalization: normalize across d_k dimension
        Q = self.q_norm(Q)  # Applied per head, per position
        K = self.k_norm(K)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(out)

# Compare attention score distributions: with vs without QK-norm
d_model, n_heads, T, B = 128, 8, 20, 4
qk_attn = QKNormAttention(d_model, n_heads)

# Simulate trained weights with large norms
with torch.no_grad():
    qk_attn.W_q.weight.data *= 5  # Simulate grown weights
    qk_attn.W_k.weight.data *= 5

x_test = torch.randn(B, T, d_model)
# Standard attention scores (manually)
Q_raw = qk_attn.W_q(x_test).view(B, T, n_heads, d_model//n_heads).transpose(1,2)
K_raw = qk_attn.W_k(x_test).view(B, T, n_heads, d_model//n_heads).transpose(1,2)
scores_raw = torch.matmul(Q_raw, K_raw.transpose(-2,-1)) / math.sqrt(d_model//n_heads)

# QK-normed scores
Q_norm = qk_attn.q_norm(Q_raw)
K_norm = qk_attn.k_norm(K_raw)
scores_normed = torch.matmul(Q_norm, K_norm.transpose(-2,-1)) / math.sqrt(d_model//n_heads)

print(f"Raw attention scores    — max: {scores_raw.abs().max().item():.2f}, std: {scores_raw.std().item():.2f}")
print(f"QK-normed scores        — max: {scores_normed.abs().max().item():.2f}, std: {scores_normed.std().item():.2f}")
print("QK-norm bounds logit scale regardless of weight magnitude")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| BatchNorm | Normalizes over batch+spatial; breaks at batch=1; not used in transformers |
| LayerNorm | Normalizes over $d_{\text{model}}$ per token; batch-independent; standard in BERT/GPT |
| RMSNorm | Removes mean centering; $x / \text{RMS}(x)$; 20-40% faster; LLaMA/Mistral/Gemma |
| Post-LN | Original transformer; LN after residual; activations grow with depth; needs warmup |
| Pre-LN | Modern standard; LN before sublayer; stable scale across depth; no warmup needed |
| QK-Norm | Normalize Q,K before attention; prevents attention logit explosion at scale |
| Learnable affine | $\gamma, \beta$ in LayerNorm; $\gamma$ only in RMSNorm; learned per-dimension scale |
| Normalization placement | First thing to check when a deep transformer doesn't converge |